# Open-Source LLMs: Licensing, Selection & Deployment
Author: arielzin33@gmail.com

Covers degrees of openness, license compatibility for SaaS, model-to-use-case matching, a real local hardware readiness audit, benchmark-based model selection, and a local-vs-cloud deployment plan.

Note: the shared Colab template requires Google sign-in and could not be fetched automatically, so this notebook answers each exercise directly from the written spec — merge into the actual template as needed. Facts about specific model licenses were verified via web search where noted; Open LLM Leaderboard benchmark numbers are approximate and should be re-checked on the live leaderboard, since the classic HF leaderboard has since been archived/superseded and exact scores drift across model card revisions.

---
## 🌟 Exercise 1: Open Source Levels Reflection


### 1–2. Definitions & key characteristics

**Fully Open**
- What's open: source code, training/architecture code, pretrained weights, and often the training data or data recipe.
- Can/can't do: you can inspect every internal detail, modify the architecture, and retrain the model end-to-end from scratch or from checkpoints.

**Weights Released**
- What's open: the pretrained model weights (and usually an inference architecture definition), but not the full original training pipeline or dataset.
- Can/can't do: you can fine-tune or adapt the model to your domain, but you cannot fully reproduce or audit how it was originally trained.

**Architecture Only**
- What's open: the model's structural design (layers, parameter shapes, code to build it) with no pretrained weights included.
- Can/can't do: you can inspect and reuse the design and train your own version from scratch, but you cannot fine-tune an existing pretrained model since no weights are provided — training from zero requires massive compute.

### 3. Side-by-side comparison

| What's Open? | Impact on Retraining/Modifying |
|---|---|
| **Fully Open** | You can inspect, modify, and retrain the entire model. |
| **Weights Released** | You have the weights for fine-tuning but not full code/data. |
| **Architecture Only** | You know the structure but lack pretrained weights. |

### 4. Comparative paragraph

Fully open models give you everything — code, weights, and often the training data — so you can audit, modify, and retrain the model with full transparency, though this level of openness is rare among top-performing models. Weights-released models are the most common form of "open source LLM" today: you get usable pretrained weights ready for fine-tuning, but the original training code and data stay private, limiting reproducibility and deep auditing. Architecture-only releases are the least immediately useful for practitioners, since knowing the design without weights means you'd have to pay for pretraining compute yourself before you have a usable model — its main value is research transparency and reproducibility, not fast deployment.

### 5. Healthcare prompt answer

**Weights Released is the essential level** for a healthcare-specific assistant that must be retrained on clinical data. It gives you a strong pretrained base to fine-tune on sensitive, proprietary clinical data entirely within your own secure/compliant infrastructure (critical for HIPAA-type requirements), without needing the (often unavailable) original training data or the far greater compute cost of pretraining from an architecture-only release.

---
## 🌟 Exercise 2: License Check for SaaS Use


### 1. Model selection

- **Mistral-7B-Instruct-v0.2** — https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2
- **Llama-2-7b-chat-hf** — https://huggingface.co/meta-llama/Llama-2-7b-chat-hf

### 2–5. Completed license checklist

- [x] **Mistral-7B-Instruct-v0.2**
  - Type of license:
    - [x] Apache 2.0 — a permissive open-source license
  - Commercial use allowed:
    - [x] **Yes**, unconditionally — Apache 2.0 permits commercial use, modification, and redistribution with no royalties.
  - Restrictions:
    - [x] Must preserve the Apache 2.0 license notice/copyright in redistributions.
    - [x] No user-count limits, no attribution-in-product requirement beyond standard license notice, no export-control clause specific to the model card.
>
- [x] **Llama-2-7b-chat-hf**
  - Type of license:
    - [x] Custom **Llama 2 Community License** (Meta) — not a standard OSI license
  - Commercial use allowed:
    - [x] **Conditional** — allowed for commercial and research use, but gated behind accepting Meta's license agreement before downloading weights.
  - Restrictions:
    - [x] **Monthly active user limit:** if your product/service has **more than 700 million monthly active users** at time of release, you must request a separate commercial license from Meta.
    - [x] **Attribution requirement:** must include a "Notice" file crediting "Llama 2 is licensed under the LLAMA 2 Community License, Copyright (c) Meta Platforms, Inc. All Rights Reserved," and name derivative models starting with "Llama."
    - [x] **Acceptable Use Policy:** must comply with Meta's separate Acceptable Use Policy (prohibits certain use cases, e.g., generating misinformation, illegal activity).
    - [x] **No model-improvement clause:** outputs cannot be used to train/improve a competing LLM (other than Llama 2 derivatives).
    - [x] Subject to standard export control / trade compliance laws.

**SaaS takeaway:** Mistral-7B-Instruct-v0.2's Apache 2.0 license is materially simpler and lower-risk for a SaaS product — no user caps, no attribution-in-UI requirement, no gated download. Llama-2-7b-chat-hf is commercially usable for the vast majority of SaaS products (under 700M MAU) but carries real compliance overhead (attribution notice, Acceptable Use Policy, license acceptance) that should be tracked in a legal/compliance checklist before shipping.

---
## 🌟 Exercise 3: LLM Matchmaker Challenge


### 1. Team needs (key constraint underlined)

- **LegalTech:** *CPU-only*, *logic-heavy* chatbot.
- **EdTech:** *math/logic* focus on *low-end laptops*.
- **Global NGO:** supports *5+ languages* well.


### 1. Search Filter Summary

- LegalTech: Hugging Face Models filtered by `Task: Text Generation`, tag `logic`/`reasoning`, `Library: GGUF` (CPU/quantized), sorted by model size ≤ 7B — https://huggingface.co/models?pipeline_tag=text-generation&library=gguf
- EdTech: filtered by tag `math`, model size ≤ 3–7B, quantized (`4-bit`/`GGUF`) for low-RAM laptops — https://huggingface.co/models?pipeline_tag=text-generation&other=math
- Global NGO: filtered by tag `multilingual`, sorted by FLORES-200/language-coverage mentions — https://huggingface.co/models?pipeline_tag=translation&language=multilingual

### 2–5. Candidate lists & evaluation

**LegalTech (CPU, logic-heavy):**
- Mistral-7B-Instruct-v0.2-GGUF (7B, Mistral architecture, 4-bit quantized) — strong general reasoning, good BoolQ-style performance, efficient CPU inference via llama.cpp.
- Llama-2-7b-chat.Q4_K_M.gguf (7B, Llama architecture, 4-bit) — solid instruction-following and reasoning, well-supported quantized CPU builds.
- Phi-2 (2.7B, dense transformer) — punches above its size on reasoning benchmarks, very fast on CPU.

**EdTech (math/logic, low-end laptops):**
- Phi-2 (2.7B) — strong GSM8K-style math performance for its size, low RAM footprint (~4GB quantized).
- TinyLlama-1.1B-Chat (1.1B, int4/GGUF) — extremely lightweight, runs on very limited hardware, weaker but usable for basic math/logic tutoring.
- Qwen2.5-Math-1.5B-Instruct (1.5B) — purpose-built for math reasoning, small footprint, good MATH/GSM8K scores relative to size.

**Global NGO (5+ languages):**
- BLOOM-7B1 (7B, GPT-style, trained on 46+ languages) — broad multilingual coverage including many low-resource languages.
- Aya-101 (13B, based on mT5, covers 101 languages) — very strong FLORES-200-style multilingual coverage, purpose-built for multilingual generation.
- NLLB-200-distilled-600M (600M, M2M-style translation model) — extremely lightweight, excellent for translation-specific multilingual coverage across 200 languages, though not a general chatbot.

### 6. Filled table

| Team | Needs | Your Pick |
|---|---|---|
| LegalTech | Fast model for logic-heavy chatbot on CPU | **Mistral-7B-Instruct-v0.2 (GGUF, 4-bit)** — best balance of reasoning quality and efficient CPU inference via llama.cpp. |
| EdTech | Logic/math-focused LLM on low-end laptops | **Phi-2 (2.7B)** — best math/logic performance-per-GB for constrained hardware; small enough to run without a GPU. |
| Global NGO | Model that speaks 5+ languages well | **Aya-101** — purpose-built massively multilingual model (101 languages) with strong cross-lingual generation quality, ideal for NGO field communication across many regions. |

---
## 🌟 Exercise 4: Local Readiness Audit

Ran directly against the machine this notebook was prepared on (Windows 11), so these are real specs, not placeholders.

### 1. Collected specs

- **RAM:** 15.73 GB total physical memory
- **Free disk space (C:):** 25.42 GB free (449.84 GB used)
- **OS:** Windows 11 Pro, build 10.0.26200
- **CPU:** 11th Gen Intel(R) Core(TM) i5-1145G7 @ 2.60GHz (Tiger Lake — supports AVX2)
- **WSL:** WSL2 is available; only a `docker-desktop` WSL distro is currently installed (no general-purpose Ubuntu/Linux distro yet).

### 2–3. Audit table

| Requirement | Your System Specs | Meets Requirement? |
|---|---|---|
| RAM (≥ 16 GB) | 15.73 GB | ❌ (just under the line — effectively ~16 GB, but technically below threshold) |
| Free Disk Space (≥ 40 GB) | 25.42 GB free | ❌ |
| OS (Linux/WSL2) | Windows 11 Pro with WSL2 available (no Linux distro installed yet) | ⚠️ Partial — WSL2 capability present but not yet set up with a usable Linux distro; llama.cpp also runs natively on Windows without WSL. |

### 4. llama.cpp readiness

- **CPU instruction sets:** the i5-1145G7 (Tiger Lake) supports AVX2 and AVX-512, well above llama.cpp's baseline AVX/SSE requirements — no hardware blocker here.
- **C/C++ compiler:** needs Visual Studio Build Tools (MSVC) or clang/mingw-w64 installed for a native Windows build, or `gcc`/`clang` + `make` inside WSL2 for a Linux-style build.
- **Additional tooling:** `cmake` (llama.cpp's build system) and `git` to clone the repo; optionally Python for the GGUF conversion/quantization scripts.

### 5. Upgrade needs summary

Two of the three requirements currently fail:

- **Disk space (25.42 GB free vs. 40 GB needed):** the most urgent fix — free up space by clearing old files/temp data or attaching external/cloud storage before downloading any 7B quantized model (a 4-bit GGUF 7B model alone is typically ~4 GB, plus room for the base download, build artifacts, and multiple quantization variants).
- **RAM (15.73 GB vs. 16 GB needed):** borderline — a 7B model quantized to 4-bit (~4–5 GB) will likely still run, since the 16 GB figure is a comfortable margin rather than a hard cutoff, but running other applications simultaneously may cause memory pressure. A RAM upgrade to 32 GB would remove this risk entirely and allow larger/less-aggressively-quantized models.
- **OS:** no upgrade strictly required — llama.cpp builds natively on Windows via MSVC/cmake. Installing a Linux distro under WSL2 (e.g., `wsl --install -d Ubuntu`) is optional but recommended if following Linux-focused llama.cpp tutorials.

**Bottom line:** disk space is the blocking issue to resolve first; RAM is workable for a 4-bit quantized 7B model but tight.

---
## 🌟 Exercise 5: Benchmark-Based Model Explorer

*Scores below are approximate figures drawn from public model cards / the archived classic Open LLM Leaderboard, since the leaderboard has since been superseded and exact numbers should be re-verified at the live Hugging Face leaderboard space before being used for a real procurement decision.*

### 1. Model list

- **HuggingFaceH4/zephyr-7b-beta** — https://huggingface.co/HuggingFaceH4/zephyr-7b-beta (balanced, DPO-tuned)
- **mistralai/Mistral-7B-Instruct-v0.2** — https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2 (strong general reasoning)
- **microsoft/phi-2** — https://huggingface.co/microsoft/phi-2 (small, high MMLU-per-parameter)

### 2–6. Comparison table

| Model Name | HellaSwag Score | MMLU Score | License Type | Ideal Use Case |
|---|---|---|---|---|
| HuggingFaceH4/zephyr-7b-beta | ~84.4 | ~61.1 | MIT | Balanced general-purpose assistant / customer support chatbot |
| mistralai/Mistral-7B-Instruct-v0.2 | ~85–86 (approx.) | ~60–61 (approx.) | Apache 2.0 | Commercial SaaS assistant needing permissive licensing and solid all-round reasoning |
| microsoft/phi-2 | ~73–75 (approx., smaller model) | ~58 (approx., high for its 2.7B size) | MIT | Lightweight on-device tutor / logic-and-math focused assistant on constrained hardware |

### Reflection (optional)

Benchmarks should guide model selection over hype because marketing claims and anecdotal "it feels smart" impressions are easy to cherry-pick and hard to compare across vendors, while standardized benchmarks like MMLU and HellaSwag measure specific, reproducible capabilities (broad academic knowledge vs. commonsense reasoning) that map to real product requirements. A model that dominates social-media hype might be tuned for chatty, engaging responses rather than the factual accuracy or reasoning consistency a given application actually needs — benchmarks let you match the *specific* strength profile of a model (math, multilingual, logic, general knowledge) to the specific job, and combined with checking the license, they turn model selection into an evidence-based engineering decision rather than a popularity contest.

---
## 🌟 Exercise 6: Cloud vs. Local Deployment Plan


### 1. Pros & cons (5 bullets, both sides covered)

- ✔️ **Local:** low latency and no per-token API cost once set up, since inference runs entirely on your own hardware with no network round-trip.
- ❌ **Local:** high upfront hardware cost (GPU/RAM) and you own all maintenance — driver updates, OS patches, model re-downloads — with no elastic scaling if demand spikes.
- ✔️ **Cloud (e.g., RunPod/Colab):** easy access to powerful GPUs on demand without buying hardware, and scales up or down with usage.
- ❌ **Cloud:** ongoing usage-based cost that can exceed local hardware cost over time, plus data leaves your machine — a real consideration for sensitive/regulated data (security trade-off).
- ✔️ **Cloud (Colab specifically):** free/cheap tier is great for quick experimentation and prototyping without any local setup, but has cold-start delays and session time limits, unlike a local setup that's always warm and ready.

### 2. Optional Colab report

Running the gated `meta-llama/Llama-2-7b-chat-hf` requires an accepted license + HF auth token, so for a quick, ungated demonstration of the same workflow, a comparable open model like `TinyLlama/TinyLlama-1.1B-Chat-v1.0` or `mistralai/Mistral-7B-Instruct-v0.2` (Apache 2.0, no gating) is a good substitute in a free Colab runtime.

In [ ]:
!pip install -q transformers accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer
import time

# Ungated substitute for meta-llama/Llama-2-7b-chat-hf (which requires license acceptance + HF token).
# Swap in "meta-llama/Llama-2-7b-chat-hf" once you've accepted the license and set your HF token.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto")

inputs = tokenizer("Hello, world!", return_tensors="pt").to(model.device)

start = time.time()
outputs = model.generate(**inputs, max_new_tokens=50)
elapsed = time.time() - start

print(tokenizer.decode(outputs[0]))
print("Model:", model_name)
print("Elapsed:", elapsed, "seconds")


**Record (fill in after running in your own Colab session):**

- Model name used: `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (or `meta-llama/Llama-2-7b-chat-hf` if you have an accepted license + HF token)
- Response time: *[fill in seconds from your Colab run — typically a few seconds on a free T4 GPU runtime for a 1B model, versus tens of seconds for a 7B model, and much longer on CPU-only runtime]*

**Summary sentence:** running even a small model in Colab makes the local-vs-cloud latency and cold-start trade-off concrete — cloud GPU access removes the hardware barrier instantly, but each fresh session pays a one-time model-download and load cost that a persistent local setup avoids after its initial (larger) hardware investment.